## Project Sauvabelin: Firewatch Thymio 

Welcome to our Speedy Fire Department, where we aim to keep the natural forests of Vaud safe and fireproof using our helpful Thymio Robot! We gladly present Department 51 below: 

- Soham (Responsible for Code Scaffolding, Map Generation, State Machine)
- Paul (Responsible for Path Planning)
- Remi (Responsible for Pose Estimation)
- Sébastien (Responsible for Local Navigation)

#### 0. Initial Setup

Add logging to be able to filter between logs for debugging and testing versus logging for information during execution.
Setup a `DEBUG` flag that will also help in providing extra visual information on the system, through plots and diagrams and helpful insights. 

In [ ]:
import sys
from loguru import logger

DEBUG = True
CAMERA_INDEX = 1

logger.remove()
logger.add(sys.stderr, level="DEBUG" if DEBUG else "INFO")

#### 1. Map Generation

In [ ]:
from map_handler import MapHandler
from data_types import Map
import cv2

map_handler = MapHandler(debug=DEBUG, map_tag_ids=[0,1,2,3], robot_tag_id=4)
map: Map | None  = None
cam = cv2.VideoCapture(CAMERA_INDEX)

while True:
    check, frame = cam.read()
    if not check:
        logger.error("Issue while reading camera frame. Exiting...")
        break
    cv2.imshow('Camera', frame)

    if cv2.waitKey(1) == ord('m'):
        map = map_handler.generate(
            cell_resolution=2, 
            map_width=60, 
            map_height=80, 
            frame=frame,
            dilation_margin=2)

    if cv2.waitKey(1) == ord('p'):
        map_handler.get_robot_state(robot_tag_id=4, frame=frame)

    if cv2.waitKey(1) == ord('q'):
        logger.info("Closing VideoCapture. Exiting...")
        break

cam.release()
cv2.destroyAllWindows()

#### 2a. Path Planning


In [ ]:
from path_planning import PathPlanner

if map is None:
    logger.info("Map object is None, was not generated correctly.")
    exit(1)

path_planner = PathPlanner(debug=DEBUG)
path, explored, op_count = path_planner.plan(map=map, prune=True, lookahead_distance=7)

In [ ]:
print(path)

#### 2b. Path Planning (Pruning)

#### 3a. Thymio Connection & Sensor/Command Bridge (WIP, needs testing)

Connect to the Thymio via `ClientAsync`, flash an on-robot event listener (`actuate`) to reliably apply motor speeds and top LEDs, and define helper functions to read `SensorData` and dispatch `Command` objects.

In [ ]:
import cv2
from tdmclient import ClientAsync
from data_types import Command, SensorData

client = ClientAsync()
node = await client.wait_for_node()
await node.lock()

# Register custom event: [left_speed, right_speed, led_r, led_g, led_b]
await node.register_events([("actuate", 5)])

robot_listener_program = """
onevent actuate
    motor.left.target = event.args[0]
    motor.right.target = event.args[1]
    leds.top = [event.args[2], event.args[3], event.args[4]]
"""
compile_err = await node.compile(robot_listener_program)
if compile_err is not None:
    logger.error(f"Failed to compile Thymio listener program: {compile_err}")
await node.run()

# Wait once for sensor variables to populate on node.v
await node.wait_for_variables({
    "prox.horizontal",
    "prox.ground.delta",
    "motor.left.speed",
    "motor.right.speed",
})
logger.info("Thymio connected and ready.")


def read_sensor_data(node, map_generator: MapHandler, cam: cv2.VideoCapture, dt: float) -> SensorData:
    """Collect Thymio onboard sensors and overhead camera ArUco pose into SensorData."""
    ok, frame = cam.read()
    pose_from_camera = map_generator.get_robot_state(frame=frame) if ok else None

    return SensorData(
        prox_horizontal=list(node.v.prox.horizontal),
        prox_ground=list(node.v.prox.ground.delta),
        motor_speeds=[
            int(node.v.motor.left.speed),
            int(node.v.motor.right.speed),
        ],
        dt=dt,
        pose_from_camera=pose_from_camera,
    )


async def apply_command(node, cmd: Command) -> None:
    """Convert a Command dataclass into an event sent to the Thymio."""
    r, g, b = cmd.leds_top
    await node.send_events({
        "actuate": [
            int(cmd.motor_left_speed),
            int(cmd.motor_right_speed),
            int(r),
            int(g),
            int(b),
        ]
    })

#### 3b. State Machine Initialization & Real-Time Execution Loop

In [ ]:
import time
from data_types import Mode
from local_avoidance import LocalAvoider
from path_following_pid import PIDController
from pose_estimation import PoseEstimator
from state_machine import COMMAND_STOP, StateMachine

LOOP_PERIOD_S = 0.1  # 10 Hz control loop matching Thymio proximity refresh rate

pose_estimator = PoseEstimator(debug=DEBUG)
path_follower = PIDController(debug=DEBUG)
path_follower.setup(path=path)
local_avoider = LocalAvoider(debug=DEBUG)

state_machine = StateMachine(
    map=map,
    map_generator=map_generator,
    path_planner=path_planner,
    pose_estimator=pose_estimator,
    path_follower=path_follower,
    local_avoider=local_avoider,
    debug=DEBUG,
)

cam = cv2.VideoCapture(0)
last_time = time.perf_counter()

try:
    while state_machine.mode not in (Mode.GOAL_REACHED, Mode.ERROR):
        now = time.perf_counter()
        dt = now - last_time
        last_time = now

        sensor_data = read_sensor_data(node, map_generator, cam, dt)
        cmd = state_machine.step(sensor_data)
        await apply_command(node, cmd)

        elapsed = time.perf_counter() - now
        await client.sleep(max(0.01, LOOP_PERIOD_S - elapsed))

    logger.info(f"Control loop finished in mode: {state_machine.mode}")
finally:
    await apply_command(node, COMMAND_STOP)
    cam.release()
    await node.stop()
    await node.unlock()